# Silver — ecommerce_rastreamento_entregas

Este notebook lê a camada Bronze de rastreamento, aplica as 10 regras de qualidade/negócio, salva a tabela Silver em Delta e registra o resumo das falhas em `squad1.dq_monitoring_logs`.

In [0]:
# Premissas:
# - Bronze existente como tabela Delta gerenciada.
# - Tabela de pedidos existente na Bronze ou Silver.
# - Logs DQ compartilhados em `squad1.dq_monitoring_logs`.

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType
)
import uuid

RUN_ID = str(uuid.uuid4())

CATALOGO_SCHEMA = "squad1"

TABELA_BRONZE_RASTREAMENTO = "squad1.bronze_ecommerce_rastreamento"
TABELA_SILVER_RASTREAMENTO = "squad1.silver_ecommerce_rastreamento"

# Ajuste o nome abaixo caso sua tabela de pedidos esteja com outro nome
TABELA_PEDIDOS_REFERENCIA = "squad1.silver_ecommerce_pedidos"

DQ_LOGS_TABLE = "squad1.dq_monitoring_logs"

NOME_TABELA_DQ = "silver_ecommerce_rastreamento"

##  Garantir existência da tabela dq_monitoring_logs

In [0]:
schema_dq_logs = StructType([
    StructField("run_id", StringType(), False),
    StructField("tabela", StringType(), False),
    StructField("regra", StringType(), False),
    StructField("status", StringType(), False),
    StructField("severidade", StringType(), False),
    StructField("qtd_registros_falhos", IntegerType(), False),
    StructField("qtd_registros_total", IntegerType(), False),
    StructField("timestamp_execucao", TimestampType(), False),
    StructField("arquivo_origem", StringType(), False),
])

if not spark.catalog.tableExists(DQ_LOGS_TABLE):
    df_empty_logs = spark.createDataFrame([], schema_dq_logs)

    (
        df_empty_logs.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(DQ_LOGS_TABLE)
    )

    print(f"Tabela Delta criada: {DQ_LOGS_TABLE}")
else:
    print(f"Tabela Delta já existe: {DQ_LOGS_TABLE}")

##  Ler micro-lote novo da Bronze

In [0]:
df_bronze = spark.table(TABELA_BRONZE_RASTREAMENTO)

# Evita reprocessar arquivos já enviados para a Silver
if spark.catalog.tableExists(TABELA_SILVER_RASTREAMENTO):
    df_arquivos_processados = (
        spark.table(TABELA_SILVER_RASTREAMENTO)
        .select("bronze_source_file")
        .dropDuplicates()
    )

    df_micro_lote = (
        df_bronze
        .join(df_arquivos_processados, on="bronze_source_file", how="left_anti")
    )
else:
    df_micro_lote = df_bronze

qtd_micro_lote = df_micro_lote.count()

print(f"Registros novos para processar: {qtd_micro_lote}")

if qtd_micro_lote == 0:
    dbutils.notebook.exit("Nenhum arquivo novo para processar na Silver de rastreamento.")

##  Ler tabelas de referência

In [0]:
df_pedidos_ref = (
    spark.table(TABELA_PEDIDOS_REFERENCIA)
    .select(
        F.col("id_pedido").alias("id_pedido_ref"),
        F.col("status_pedido").alias("status_pedido_ref"),
        F.to_timestamp("dt_pedido").alias("dt_pedido_ref"),
        F.to_timestamp("dt_ultima_atualizacao_status").alias("dt_ultima_atualizacao_status_ref")
    )
    .dropDuplicates(["id_pedido_ref"])
)

##  Padronizar tipos mínimos

In [0]:
df_base = (
    df_micro_lote
    .withColumn("id_rastreamento", F.trim(F.col("id_rastreamento").cast("string")))
    .withColumn("id_pedido_ecommerce", F.trim(F.col("id_pedido_ecommerce").cast("string")))
    .withColumn("status_entrega", F.lower(F.trim(F.col("status_entrega"))))
    .withColumn("codigo_rastreio", F.upper(F.trim(F.col("codigo_rastreio"))))
    .withColumn("id_transportadora", F.trim(F.col("id_transportadora").cast("string")))
    .withColumn("dt_evento", F.to_timestamp(F.col("dt_evento")))
)

##  Enriquecer com cálculos auxiliares

In [0]:
status_validos = [
    "em separacao",
    "coletado",
    "em transito",
    "saiu para entrega",
    "entregue"
]

status_ordem_expr = (
    F.when(F.col("status_entrega") == "em separacao", F.lit(1))
     .when(F.col("status_entrega") == "coletado", F.lit(2))
     .when(F.col("status_entrega") == "em transito", F.lit(3))
     .when(F.col("status_entrega") == "saiu para entrega", F.lit(4))
     .when(F.col("status_entrega") == "entregue", F.lit(5))
     .otherwise(F.lit(None).cast("int"))
)

w_id_rastreamento = Window.partitionBy("id_rastreamento")
w_pedido_status = Window.partitionBy("id_pedido_ecommerce").orderBy("status_ordem")
w_pedido = Window.partitionBy("id_pedido_ecommerce")

# O Spark não suporta countDistinct dentro de Window.
# Por isso, calculamos a quantidade distinta de transportadoras por pedido via groupBy
# e fazemos join no DataFrame principal.
df_qtd_transportadoras = (
    df_base
    .groupBy("id_pedido_ecommerce")
    .agg(
        F.countDistinct("id_transportadora").alias("qtd_transportadoras_pedido")
    )
)

df_enriquecido = (
    df_base
    .withColumn("qtd_id_rastreamento", F.count("*").over(w_id_rastreamento))
    .withColumn("status_ordem", status_ordem_expr)
    .join(
        df_pedidos_ref,
        df_base["id_pedido_ecommerce"] == df_pedidos_ref["id_pedido_ref"],
        "left"
    )
    .join(
        df_qtd_transportadoras,
        on="id_pedido_ecommerce",
        how="left"
    )
    .withColumn("qtd_transportadoras_pedido", F.coalesce(F.col("qtd_transportadoras_pedido"), F.lit(0)))
    .withColumn("pedido_existe", F.col("id_pedido_ref").isNotNull())
    .withColumn("dt_evento_status_anterior", F.lag("dt_evento").over(w_pedido_status))
    .withColumn(
        "tem_evento_entregue_pedido",
        F.max(F.when(F.col("status_entrega") == "entregue", F.lit(1)).otherwise(F.lit(0))).over(w_pedido)
    )
    .withColumn(
        "dt_coletado_pedido",
        F.min(F.when(F.col("status_entrega") == "coletado", F.col("dt_evento"))).over(w_pedido)
    )
    .withColumn(
        "dt_entregue_pedido",
        F.max(F.when(F.col("status_entrega") == "entregue", F.col("dt_evento"))).over(w_pedido)
    )
)

display(df_enriquecido.limit(10))

##  Aplicar as 10 regras em colunas booleanas

In [0]:
regex_codigo_rastreio = r"^[A-Z]{2}[0-9]{9}$"

df_silver = (
    df_enriquecido
    .withColumn(
        "r1_id_rastreamento_falhou",
        F.col("id_rastreamento").isNull() | (F.col("id_rastreamento") == "") | (F.col("qtd_id_rastreamento") > 1)
    )
    .withColumn(
        "r2_status_entrega_falhou",
        ~F.col("status_entrega").isin(status_validos)
    )
    .withColumn(
        "r3_id_pedido_ecommerce_falhou",
        F.col("id_pedido_ecommerce").isNull() | (F.col("id_pedido_ecommerce") == "") | (~F.col("pedido_existe"))
    )
    .withColumn(
        "r4_dt_evento_falhou",
        F.col("dt_evento").isNull() | (F.col("dt_evento") > F.current_timestamp())
    )
    .withColumn(
        "r5_codigo_rastreio_falhou",
        F.col("codigo_rastreio").isNull() | (~F.col("codigo_rastreio").rlike(regex_codigo_rastreio))
    )
    .withColumn(
        "r6_ordem_cronologica_status_falhou",
        F.col("dt_evento_status_anterior").isNotNull() & (F.col("dt_evento") < F.col("dt_evento_status_anterior"))
    )
    .withColumn(
        "r7_pedido_entregue_sem_evento_entregue_falhou",
        (F.col("status_pedido_ref") == "Entregue") & (F.col("tem_evento_entregue_pedido") == 0)
    )
    .withColumn(
        "r8_sla_coletado_entregue_falhou",
        F.col("dt_coletado_pedido").isNotNull()
        & F.col("dt_entregue_pedido").isNotNull()
        & (F.datediff(F.col("dt_entregue_pedido"), F.col("dt_coletado_pedido")) > 30)
    )
    .withColumn(
        "r9_pedido_cancelado_com_rastreamento_falhou",
        F.col("status_pedido_ref") == "Cancelado"
    )
    .withColumn(
        "r10_transportadora_inconsistente_falhou",
        (F.col("status_pedido_ref") != "Cancelado") & (F.col("qtd_transportadoras_pedido") != 1)
    )
    .withColumn(
        "silver_linha_valida",
        ~(
            F.col("r1_id_rastreamento_falhou")
            | F.col("r2_status_entrega_falhou")
            | F.col("r3_id_pedido_ecommerce_falhou")
            | F.col("r4_dt_evento_falhou")
            | F.col("r5_codigo_rastreio_falhou")
            | F.col("r6_ordem_cronologica_status_falhou")
            | F.col("r7_pedido_entregue_sem_evento_entregue_falhou")
            | F.col("r8_sla_coletado_entregue_falhou")
            | F.col("r9_pedido_cancelado_com_rastreamento_falhou")
            | F.col("r10_transportadora_inconsistente_falhou")
        )
    )
    .withColumn("silver_processed_at", F.current_timestamp())
)

## Salvar Silver Delta

In [0]:
colunas_tecnicas_remover = [
    "qtd_id_rastreamento",
    "id_pedido_ref",
    "status_pedido_ref",
    "dt_pedido_ref",
    "dt_ultima_atualizacao_status_ref",
    "pedido_existe",
    "status_ordem",
    "dt_evento_status_anterior",
    "qtd_transportadoras_pedido",
    "tem_evento_entregue_pedido",
    "dt_coletado_pedido",
    "dt_entregue_pedido"
]

df_silver_final = df_silver.drop(*[c for c in colunas_tecnicas_remover if c in df_silver.columns])

(
    df_silver_final.write
    .format("delta")
    .mode("append")
    .saveAsTable(TABELA_SILVER_RASTREAMENTO)
)

print(f"Silver gravada: {TABELA_SILVER_RASTREAMENTO}")

##  Criar logs DQ por regra e arquivo

In [0]:
def criar_log_regra(df, nome_regra, coluna_flag, severidade):
    return (
        df
        .groupBy("bronze_source_file")
        .agg(
            F.count("*").cast("int").alias("qtd_registros_total"),
            F.sum(F.when(F.col(coluna_flag), 1).otherwise(0)).cast("int").alias("qtd_registros_falhos")
        )
        .withColumn("run_id", F.lit(RUN_ID))
        .withColumn("tabela", F.lit(NOME_TABELA_DQ))
        .withColumn("regra", F.lit(nome_regra))
        .withColumn(
            "status",
            F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL")).otherwise(F.lit("PASS"))
        )
        .withColumn("severidade", F.lit(severidade))
        .withColumn("timestamp_execucao", F.current_timestamp())
        .withColumnRenamed("bronze_source_file", "arquivo_origem")
        .select(
            "run_id",
            "tabela",
            "regra",
            "status",
            "severidade",
            "qtd_registros_falhos",
            "qtd_registros_total",
            "timestamp_execucao",
            "arquivo_origem"
        )
    )

df_logs = (
    criar_log_regra(df_silver, "R1 - id_rastreamento não pode ser nulo nem duplicado", "r1_id_rastreamento_falhou", "Critica")
    .unionByName(criar_log_regra(df_silver, "R2 - status_entrega deve estar na lista permitida", "r2_status_entrega_falhou", "Critica"))
    .unionByName(criar_log_regra(df_silver, "R3 - id_pedido_ecommerce obrigatório e existente em pedidos", "r3_id_pedido_ecommerce_falhou", "Critica"))
    .unionByName(criar_log_regra(df_silver, "R4 - dt_evento não pode ser nula nem futura", "r4_dt_evento_falhou", "Critica"))
    .unionByName(criar_log_regra(df_silver, "R5 - codigo_rastreio deve seguir padrão 2 letras + 9 dígitos", "r5_codigo_rastreio_falhou", "Aviso"))
    .unionByName(criar_log_regra(df_silver, "R6 - dt_evento deve respeitar ordem cronológica dos status", "r6_ordem_cronologica_status_falhou", "Critica"))
    .unionByName(criar_log_regra(df_silver, "R7 - pedido entregue deve ter evento entregue", "r7_pedido_entregue_sem_evento_entregue_falhou", "Critica"))
    .unionByName(criar_log_regra(df_silver, "R8 - tempo entre coletado e entregue deve ser <= 30 dias", "r8_sla_coletado_entregue_falhou", "Aviso"))
    .unionByName(criar_log_regra(df_silver, "R9 - pedidos cancelados não devem ter rastreamento", "r9_pedido_cancelado_com_rastreamento_falhou", "Critica"))
    .unionByName(criar_log_regra(df_silver, "R10 - pedido não cancelado deve ter exatamente 1 transportadora", "r10_transportadora_inconsistente_falhou", "Critica"))
)

##  Gravar logs em squad1.dq_monitoring_logs

In [0]:
# Evita duplicidade de logs para o mesmo arquivo/tabela/regra
df_logs_existentes = (
    spark.table(DQ_LOGS_TABLE)
    .select("tabela", "regra", "arquivo_origem")
    .dropDuplicates()
)

df_logs_novos = (
    df_logs
    .join(
        df_logs_existentes,
        on=["tabela", "regra", "arquivo_origem"],
        how="left_anti"
    )
)

qtd_logs_novos = df_logs_novos.count()
print(f"Logs novos a gravar: {qtd_logs_novos}")

if qtd_logs_novos > 0:
    (
        df_logs_novos.write
        .format("delta")
        .mode("append")
        .saveAsTable(DQ_LOGS_TABLE)
    )

print(f"Logs DQ gravados em: {DQ_LOGS_TABLE}")

##  Validação final

In [0]:
print("Resumo da Silver de rastreamento:")
display(
    df_silver_final
    .groupBy("silver_linha_valida")
    .count()
)

print("Resumo de logs gerados:")
display(df_logs.orderBy("arquivo_origem", "regra"))